[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/04_reporting/D4_hcd_apr_tables.ipynb)

# D4: HCD Annual Progress Report (APR) Tables

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Understand the APR** - what it is and why cities must file it
2. **Map local data to APR fields** - connect Berkeley's data to state requirements
3. **Generate an APR table** - create a structured output ready for reporting
4. **Identify gaps** - understand what's missing and what's hard to automate

---

## What is the Annual Progress Report (APR)?

California law requires every city and county to submit an **Annual Progress Report** to the Department of Housing and Community Development (HCD). This report tracks:

- **Housing permits issued** during the year
- **Units built** and their affordability levels
- **Progress toward RHNA** (Regional Housing Needs Allocation)

### Why It Matters

| Stakeholder | Why They Care |
|-------------|---------------|
| **State (HCD)** | Monitors if cities are meeting housing goals |
| **Cities** | Compliance with state law; affects funding |
| **Residents** | Transparency on housing production |
| **Researchers** | Data for housing policy analysis |

### The APR Tables

The APR consists of several tables:

| Table | What It Reports |
|-------|----------------|
| **Table A** | Housing development applications |
| **Table A2** | Annual building activity (permits issued) |
| **Table B** | RHNA progress |
| **Table C** | Sites inventory |
| **Table D** | Program implementation |

This notebook focuses on **Table A2** - the most data-intensive table.

---

## 1. Setup

In [ ]:
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# Find project root
def find_project_root():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    return current

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config
config_path = ROOT / '00_config/berkeley_config.json'
if config_path.exists():
    with open(config_path) as f:
        CONFIG = json.load(f)
    DATA_DIR = ROOT / CONFIG['paths']['data_dir']
    OUTPUT_DIR = ROOT / CONFIG['paths']['output_dir']
else:
    DATA_DIR = ROOT / 'data/processed'
    OUTPUT_DIR = ROOT / 'data/outputs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Load Housing Projects Data

In [ ]:
# Load the main housing projects dataset
housing_path = DATA_DIR / 'housing_projects_FINAL.csv'

if housing_path.exists():
    df = pd.read_csv(housing_path)
    print(f"Loaded {len(df)} housing projects")
    print(f"\nColumns available: {df.columns.tolist()}")
else:
    print(f"File not found: {housing_path}")
    df = None

In [ ]:
# Preview the data
if df is not None:
    display(df.head())

## 3. APR Table A2 Field Mapping

Table A2 requires specific fields. Here's how our data maps:

| APR Field | Our Field | Notes |
|-----------|-----------|-------|
| Project Identifier | `id` or `apn` | Unique ID for each project |
| Street Address | `address_display` | Project location |
| Assessor Parcel Number | `apn` | County parcel ID |
| Unit Category | `unit_category` | SFD, 2-4 units, 5+ units, ADU |
| Tenure | `tenure` | Owner or Renter |
| Very Low Income | `vli_units` | Units affordable to <50% AMI |
| Low Income | `li_units` | Units affordable to 50-80% AMI |
| Moderate Income | `mod_units` | Units affordable to 80-120% AMI |
| Above Moderate | `above_mod_units` | Market rate units |
| Total Units | `net_units` | Total new units |
| Entitlement Date | `entitlement_date` | When approved |
| Building Permit Date | `building_permit_date` | When BP issued |
| Certificate of Occupancy | `co_issued_date` | When completed |

In [ ]:
# Check which APR fields we have data for
if df is not None:
    apr_field_mapping = {
        'Project ID': ['id'],
        'Street Address': ['address_display'],
        'APN': ['apn'],
        'Unit Category': ['unit_category', 'project_size_category'],
        'Tenure': ['tenure'],
        'Total Units': ['net_units', 'new_units'],
        'VLI Units': ['vli_units', 'vli_units_extracted'],
        'Low Income Units': ['li_units'],
        'Moderate Units': ['mod_units'],
        'Above Moderate': ['above_mod_units'],
        'Year': ['year'],
        'Status': ['status'],
    }
    
    print("APR Field Coverage:")
    print("=" * 50)
    
    for apr_field, possible_cols in apr_field_mapping.items():
        found = None
        for col in possible_cols:
            if col in df.columns:
                non_null = df[col].notna().sum()
                pct = 100 * non_null / len(df)
                found = f"{col} ({pct:.0f}% populated)"
                break
        
        status = found if found else "MISSING"
        print(f"  {apr_field:20} -> {status}")

## 4. Generate APR Table A2 (Approximation)

Let's create a simplified version of Table A2 with the data we have.

In [ ]:
def generate_apr_table_a2(df, year=None):
    """
    Generate an approximation of APR Table A2.
    
    Table A2: Annual Building Activity Report Summary
    Reports housing units permitted during the reporting year.
    """
    # Filter by year if specified
    if year and 'year' in df.columns:
        df_year = df[df['year'] == year].copy()
    else:
        df_year = df.copy()
    
    # Build the APR table structure
    apr_records = []
    
    for idx, row in df_year.iterrows():
        # Determine unit category (APR categories)
        units = row.get('net_units', 0) or 0
        if units == 1:
            unit_cat = 'SFD'  # Single Family Detached
        elif units <= 4:
            unit_cat = '2-4 Units'
        else:
            unit_cat = '5+ Units'
        
        # Check for ADU in description
        desc = str(row.get('description', '')).upper()
        if 'ADU' in desc or 'ACCESSORY' in desc:
            unit_cat = 'ADU'
        
        # Income breakdown (use available data or default to market rate)
        vli = row.get('vli_units_extracted', 0) or 0
        li = row.get('li_units', 0) or 0
        mod = row.get('mod_units', 0) or 0
        above_mod = max(0, units - vli - li - mod)
        
        apr_record = {
            'project_id': row.get('id', idx),
            'street_address': row.get('address_display', ''),
            'apn': row.get('apn', ''),
            'unit_category': unit_cat,
            'tenure': row.get('tenure', 'Renter'),  # Default to renter for multi-family
            'vli_units': int(vli),
            'li_units': int(li),
            'mod_units': int(mod),
            'above_mod_units': int(above_mod),
            'total_units': int(units),
            'year': row.get('year', ''),
            'status': row.get('status', ''),
        }
        apr_records.append(apr_record)
    
    return pd.DataFrame(apr_records)


# Generate the table
if df is not None:
    apr_table = generate_apr_table_a2(df)
    print(f"Generated APR Table A2 with {len(apr_table)} projects")
    display(apr_table.head(10))

In [ ]:
# Summary statistics (like APR Table B - RHNA progress)
if df is not None and len(apr_table) > 0:
    print("APR Summary Statistics")
    print("=" * 50)
    
    # Units by income category
    print("\nUnits by Income Category:")
    print(f"  Very Low Income:  {apr_table['vli_units'].sum():>6}")
    print(f"  Low Income:       {apr_table['li_units'].sum():>6}")
    print(f"  Moderate Income:  {apr_table['mod_units'].sum():>6}")
    print(f"  Above Moderate:   {apr_table['above_mod_units'].sum():>6}")
    print(f"  " + "-"*25)
    print(f"  TOTAL:            {apr_table['total_units'].sum():>6}")
    
    # Units by category
    print("\nUnits by Structure Type:")
    by_category = apr_table.groupby('unit_category')['total_units'].sum()
    for cat, units in by_category.items():
        print(f"  {cat:15} {units:>6}")

## 5. Save the APR Table

In [ ]:
# Save to CSV
if df is not None and len(apr_table) > 0:
    output_path = OUTPUT_DIR / 'apr_table_A2_example.csv'
    apr_table.to_csv(output_path, index=False)
    print(f"Saved APR Table A2 to: {output_path}")
    print(f"Records: {len(apr_table)}")
    print(f"Total units: {apr_table['total_units'].sum()}")

## 6. What's Hard About Automating Today's APR Form

The current APR process has several challenges that make full automation difficult:

### Challenge 1: Income Categories Aren't Tracked
- APR requires breakdown by VLI/LI/Moderate/Above Moderate
- Most permit systems don't capture affordability at time of permit
- Affordable housing projects often tracked separately

**Discussion:** How could cities better track affordability data at permit issuance?

### Challenge 2: Date Fields Are Inconsistent
- APR asks for entitlement date, building permit date, CO date
- Different permit systems record different milestones
- Some dates are in free-text fields, not standardized

**Discussion:** What's the minimum set of dates a permit system should require?

### Challenge 3: Unit Categories Require Interpretation
- Is a duplex "SFD" or "2-4 Units"?
- What about a house with an ADU?
- Mixed-use buildings with residential units?

**Discussion:** How could APR definitions be made clearer?

### Challenge 4: The Excel Template Isn't Machine-Friendly
- HCD provides an Excel template with complex formatting
- Merged cells, hidden rows, validation rules
- Difficult to programmatically populate

**Discussion:** What would a better APR submission format look like? (JSON? CSV? API?)

### Challenge 5: Reporting Period Cutoffs
- APR covers January 1 - December 31
- Projects in progress at year-end require careful handling
- Some systems track fiscal years, not calendar years

**Discussion:** How should "in progress" projects be reported?

---

## Recommendations for Improvement

1. **Standardize permit data entry** at the source (require specific fields)
2. **Create an open data standard** for housing permit data
3. **Develop an APR API** for direct submission from city systems
4. **Publish mapping guides** showing how common permit systems map to APR
5. **Track affordability** at permit issuance, not just project completion

---

## Summary

This notebook:
1. Explained what the HCD Annual Progress Report is
2. Mapped Berkeley's housing data to APR Table A2 fields
3. Generated an example APR table from local data
4. Identified challenges in automating APR production

**Key Insight:** While we can automate much of the APR, full automation requires:
- Better source data (especially income categories)
- Standardized date tracking
- Clearer field definitions from HCD

**Next:** See `D1_monthly_report_generator.ipynb` for broader reporting capabilities.